# Clean HAM10000 + BCN20000 metadata for the 7-class thesis setup

This notebook builds a **clean, auditable master metadata table** for:

- `data/HAM10k`
- `data/BCN20k`

It is designed for your primary experiment:

- **7-class taxonomy**: `MEL, NV, BCC, AKIEC, BKL, DF, VASC`
- **exclude SCC** from the primary experiment
- keep **debug prints** and comments everywhere
- save cleaned outputs so you can later turn this into a `.py` script

## What this notebook does

1. Loads both `metadata.csv` files
2. Standardizes metadata columns
3. Checks image file existence
4. Normalizes diagnosis text
5. Maps diagnoses to the 7 thesis classes
6. Flags samples to keep / exclude for the primary experiment
7. Computes simple dataset audit reports
8. Computes exact image hashes for duplicate detection
9. Saves:
   - per-dataset cleaned CSVs
   - one merged master CSV
   - duplicate reports

## Expected folder layout

Each dataset folder should contain files like:

- `metadata.csv`
- `<ISIC_ID>.jpg`
- `attribution.txt`
- `licenses/`

Example:

```text
data/HAM10k/
    metadata.csv
    ISIC_0036064.jpg
    attribution.txt
    licenses/

data/BCN20k/
    metadata.csv
    ISIC_0053453.jpg
    attribution.txt
    licenses/
```


In [1]:

# =========================
# 0) Imports and settings
# =========================
from pathlib import Path
import hashlib
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# ---- Paths ----
# PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = Path("..")
DATA_DIR = REPO_ROOT / "data"
HAM_DIR = DATA_DIR / "HAM10k"
BCN_DIR = DATA_DIR / "BCN20k"
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("HAM_DIR exists:", HAM_DIR.exists(), HAM_DIR)
print("BCN_DIR exists:", BCN_DIR.exists(), BCN_DIR)
print("OUT_DIR:", OUT_DIR)

HAM_META = HAM_DIR / "metadata.csv"
BCN_META = BCN_DIR / "metadata.csv"

print("\nMetadata files:")
print("HAM_META:", HAM_META, "exists =", HAM_META.exists())
print("BCN_META:", BCN_META, "exists =", BCN_META.exists())


REPO_ROOT: ..
HAM_DIR exists: True ../data/HAM10k
BCN_DIR exists: True ../data/BCN20k
OUT_DIR: ../data/processed

Metadata files:
HAM_META: ../data/HAM10k/metadata.csv exists = True
BCN_META: ../data/BCN20k/metadata.csv exists = True


In [ ]:

# ============================================
# 1) Helper functions for cleaning and audit
# ============================================

def require_file(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

def snake_strip(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    return value

def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    return value

def normalize_lower(value):
    if pd.isna(value):
        return np.nan
    return normalize_text(value).lower()

def normalize_label(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = x.replace("&", " and ")
    x = re.sub(r"[^a-z0-9]+", " ", x) # replace punctuation with spaces
    x = re.sub(r"\s+", " ", x).strip() # collapse multiple spaces
    return x

def harmonize_primary_label(raw_label):
    norm = normalize_label(raw_label)

    if norm is None:
        return np.nan, "exclude_missing_diagnosis"

    if norm in PRIMARY_7_MAP:
        return PRIMARY_7_MAP[norm], "keep"

    if norm in PRIMARY_EXCLUDE:
        return np.nan, PRIMARY_EXCLUDE[norm]

    return np.nan, "exclude_unmapped_label"

def find_image_path(dataset_dir: Path, isic_id: str):
    jpg = dataset_dir / f"{isic_id}.jpg"
    png = dataset_dir / f"{isic_id}.png"
    if jpg.exists():
        return jpg
    if png.exists():
        return png
    return None

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def print_basic_overview(df, name):
    print(f"\n{'='*80}")
    print(f"{name} overview")
    print(f"{'='*80}")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("\nhead:")
    display(df.head(3))
    print("\nnull counts:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("n_missing").head(20))

def count_unique_summary(df, cols):
    rows = []
    for c in cols:
        if c in df.columns:
            rows.append({
                "column": c,
                "n_unique": df[c].nunique(dropna=True),
                "n_missing": int(df[c].isna().sum())
            })
    out = pd.DataFrame(rows).sort_values("column").reset_index(drop=True)
    return out

def first_existing(series, fallback=np.nan):
    vals = series.dropna()
    if len(vals) == 0:
        return fallback
    return vals.iloc[0]

# Diagnosis mapping for the primary 7-class experiment
# We use diagnosis_3 because it is the most specific diagnosis field in your examples.
PRIMARY_7_MAP = {
    # melanoma
    # "melanoma": "MEL",
    "melanoma nos": "MEL",

    # nevus
    "nevus": "NV",

    # basal cell carcinoma
    "basal cell carcinoma": "BCC",
    # "bcc": "BCC",

    # actinic keratosis family
    # "actinic keratosis": "AKIEC",
    "solar or actinic keratosis": "AKIEC",
    # "bowen disease": "AKIEC",
    # "intraepithelial carcinoma": "AKIEC",
    # "actinic keratosis bowens disease intraepithelial carcinoma": "AKIEC",

    # benign keratosis-like
    "pigmented benign keratosis": "BKL",
    "seborrheic keratosis": "BKL",
    # "seborrhoeic keratosis": "BKL",
    "solar lentigo": "BKL",
    # "lichen planus like keratosis": "BKL",
    # "lichen planus-like keratosis": "BKL",
    # "lentigo nos": "BKL",
    # "benign keratosis": "BKL",

    # dermatofibroma
    "dermatofibroma": "DF",

    # vascular
    # "vascular lesion": "VASC",
    # "angioma": "VASC",
    # "hemangioma": "VASC",
    # "haemangioma": "VASC",
    # "pyogenic granuloma": "VASC",
}

# Diagnoses to exclude from the main 7-class experiment
PRIMARY_EXCLUDE = {
    "squamous cell carcinoma": "exclude_scc",
    "squamous cell carcinoma nos": "exclude_scc",
    "scc": "exclude_scc",
    "melanoma metastasis": "exclude_melanoma_metastasis",
    "scar": "exclude_scar",
    "unknown": "exclude_unknown",
    "other": "exclude_other",
}

def load_and_standardize_metadata(meta_path: Path, dataset_dir: Path, source_name: str):
    require_file(meta_path)
    df = pd.read_csv(meta_path)

    print(f"Loaded {source_name}: {df.shape}")
    print(f"{source_name} columns:")
    print(list(df.columns))

    # Normalize column names
    df.columns = [normalize_text(c) for c in df.columns]

    # Normalize string values in object columns
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        df[c] = df[c].map(normalize_text)

    # Add source dataset and image path
    df["source_dataset"] = source_name
    df["image_path"] = df["isic_id"].map(lambda x: find_image_path(dataset_dir, x))
    df["image_exists"] = df["image_path"].notna()

    # Raw label from diagnosis_3
    if "diagnosis_3" not in df.columns:
        raise KeyError(f"{source_name}: expected column 'diagnosis_3' not found")

    df["raw_label"] = df["diagnosis_3"]
    df["raw_label_norm"] = df["raw_label"].apply(normalize_label)

    mapped = df["raw_label"].apply(harmonize_primary_label)
    df["harmonized_label"] = mapped.apply(lambda x: x[0])
    df["primary_status"] = mapped.apply(lambda x: x[1])

    # Standard boolean-like cleanup
    for c in ["concomitant_biopsy", "melanocytic"]:
        if c in df.columns:
            df[c] = df[c].replace({True: True, False: False, "True": True, "False": False})

    # Standardize sex
    if "sex" in df.columns:
        sex_map = {
            "male": "male",
            "female": "female",
            "unknown": np.nan,
            "": np.nan,
        }
        df["sex_clean"] = df["sex"].map(lambda x: sex_map.get(normalize_lower(x), normalize_lower(x) if pd.notna(x) else np.nan))
    else:
        df["sex_clean"] = np.nan

    # Standardize age
    if "age_approx" in df.columns:
        df["age_approx"] = pd.to_numeric(df["age_approx"], errors="coerce")

    # Standardize site
    if "anatom_site_general" not in df.columns:
        df["anatom_site_general"] = np.nan

    # Keep flag for the main primary experiment
    df["keep_for_primary_experiment"] = (
        df["primary_status"].eq("keep")
        & df["image_exists"]
        & ((df["image_type"] == "dermoscopic") if "image_type" in df.columns else True)
    )

    # Candidate group id for future split logic:
    # patient_id is not available in these metadata examples, so lesion_id becomes key.
    df["group_id"] = df["lesion_id"].fillna(df["isic_id"])

    # Helpful sort
    cols_front = [
        "isic_id", "source_dataset", "image_path", "image_exists",
        "lesion_id", "group_id",
        "raw_label", "harmonized_label", "primary_status", "keep_for_primary_experiment",
        "diagnosis_1", "diagnosis_2", "diagnosis_3", "diagnosis_confirm_type",
        "image_type", "melanocytic", "sex", "sex_clean", "age_approx",
        "anatom_site_general",
    ]
    cols_front = [c for c in cols_front if c in df.columns]
    cols_rest = [c for c in df.columns if c not in cols_front]
    df = df[cols_front + cols_rest]

    return df


In [3]:

# =========================
# 2) Load both datasets
# =========================
ham = load_and_standardize_metadata(HAM_META, HAM_DIR, "HAM10000")
bcn = load_and_standardize_metadata(BCN_META, BCN_DIR, "BCN20000")

print_basic_overview(ham, "HAM10000")
print_basic_overview(bcn, "BCN20000")


Loaded HAM10000: (11720, 19)
HAM10000 columns:
['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_3', 'anatom_site_general', 'anatom_site_special', 'concomitant_biopsy', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_manipulation', 'image_type', 'lesion_id', 'melanocytic', 'sex']
Loaded BCN20000: (18946, 17)
BCN20000 columns:
['isic_id', 'attribution', 'copyright_license', 'age_approx', 'anatom_site_1', 'anatom_site_2', 'anatom_site_general', 'anatom_site_special', 'concomitant_biopsy', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 'lesion_id', 'melanocytic', 'sex']

HAM10000 overview
shape: (11720, 29)
columns: ['isic_id', 'source_dataset', 'image_path', 'image_exists', 'lesion_id', 'group_id', 'raw_label', 'harmonized_label', 'primary_status', 'keep_for_primary_experiment', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 

,isic_id,source_dataset,image_path,image_exists,lesion_id,group_id,raw_label,harmonized_label,primary_status,keep_for_primary_experiment,diagnosis_1,diagnosis_2,diagnosis_3,diagnosis_confirm_type,image_type,melanocytic,sex,sex_clean,age_approx,anatom_site_general,attribution,copyright_license,anatom_site_1,anatom_site_2,anatom_site_3,anatom_site_special,concomitant_biopsy,image_manipulation,raw_label_norm
0,ISIC_0024306,HAM10000,../data/HAM10k/ISIC_0024306.jpg,True,IL_7252831,IL_7252831,Nevus,NV,keep,True,Benign,Benign melanocytic proliferations,Nevus,serial imaging showing no change,dermoscopic,True,male,male,45.0,NaN,"ViDIR Group, Department of Dermatology, Medical University of Vienna",CC-BY-NC,Trunk,NaN,NaN,NaN,False,NaN,nevus
1,ISIC_0024307,HAM10000,../data/HAM10k/ISIC_0024307.jpg,True,IL_6125741,IL_6125741,Nevus,NV,keep,True,Benign,Benign melanocytic proliferations,Nevus,serial imaging showing no change,dermoscopic,True,male,male,50.0,lower extremity,"ViDIR Group, Department of Dermatology, Medical University of Vienna",CC-BY-NC,Lower extremity,NaN,NaN,NaN,False,NaN,nevus
2,ISIC_0024308,HAM10000,../data/HAM10k/ISIC_0024308.jpg,True,IL_3692653,IL_3692653,Nevus,NV,keep,True,Benign,Benign melanocytic proliferations,Nevus,serial imaging showing no change,dermoscopic,True,female,female,55.0,NaN,"ViDIR Group, Department of Dermatology, Medical University of Vienna",CC-BY-NC,Trunk,NaN,NaN,NaN,False,NaN,nevus



null counts:


,n_missing
image_manipulation,11719
anatom_site_special,11183
anatom_site_3,9038
anatom_site_2,5804
anatom_site_general,2162
anatom_site_1,581
harmonized_label,409
age_approx,383
sex,343
sex_clean,343



BCN20000 overview
shape: (18946, 27)
columns: ['isic_id', 'source_dataset', 'image_path', 'image_exists', 'lesion_id', 'group_id', 'raw_label', 'harmonized_label', 'primary_status', 'keep_for_primary_experiment', 'diagnosis_1', 'diagnosis_2', 'diagnosis_3', 'diagnosis_confirm_type', 'image_type', 'melanocytic', 'sex', 'sex_clean', 'age_approx', 'anatom_site_general', 'attribution', 'copyright_license', 'anatom_site_1', 'anatom_site_2', 'anatom_site_special', 'concomitant_biopsy', 'raw_label_norm']

head:


,isic_id,source_dataset,image_path,image_exists,lesion_id,group_id,raw_label,harmonized_label,primary_status,keep_for_primary_experiment,diagnosis_1,diagnosis_2,diagnosis_3,diagnosis_confirm_type,image_type,melanocytic,sex,sex_clean,age_approx,anatom_site_general,attribution,copyright_license,anatom_site_1,anatom_site_2,anatom_site_special,concomitant_biopsy,raw_label_norm
0,ISIC_0053453,BCN20000,../data/BCN20k/ISIC_0053453.jpg,True,IL_0433332,IL_0433332,Nevus,NV,keep,True,Benign,Benign melanocytic proliferations,Nevus,histopathology,dermoscopic,True,male,male,60.0,anterior torso,Hospital Clínic de Barcelona,CC-BY-NC,Trunk,Anterior trunk,NaN,True,nevus
1,ISIC_0053454,BCN20000,../data/BCN20k/ISIC_0053454.jpg,True,IL_8314504,IL_8314504,"Melanoma, NOS",MEL,keep,True,Malignant,Malignant melanocytic proliferations (Melanoma),"Melanoma, NOS",histopathology,dermoscopic,True,male,male,55.0,anterior torso,Hospital Clínic de Barcelona,CC-BY-NC,Trunk,Anterior trunk,NaN,True,melanoma nos
2,ISIC_0053455,BCN20000,../data/BCN20k/ISIC_0053455.jpg,True,IL_8065011,IL_8065011,NaN,NaN,exclude_missing_diagnosis,False,NaN,NaN,NaN,histopathology,dermoscopic,NaN,male,male,50.0,lower extremity,Hospital Clínic de Barcelona,CC-BY-NC,Lower extremity,NaN,NaN,True,NaN



null counts:


,n_missing
anatom_site_special,18177
anatom_site_2,11577
diagnosis_confirm_type,3506
harmonized_label,2813
raw_label,1307
diagnosis_3,1307
raw_label_norm,1307
diagnosis_1,1156
diagnosis_2,1156
melanocytic,1156


In [4]:

# ====================================
# 3) Quick audit of key metadata fields
# ====================================

audit_cols = [
    "isic_id", "lesion_id", "raw_label", "harmonized_label", "primary_status",
    "diagnosis_confirm_type", "image_type", "image_exists", "sex", "sex_clean",
    "age_approx", "anatom_site_general"
]

print("\nHAM unique/missing summary:")
display(count_unique_summary(ham, audit_cols))

print("\nBCN unique/missing summary:")
display(count_unique_summary(bcn, audit_cols))

print("\nHAM image_type distribution:")
display(ham["image_type"].value_counts(dropna=False).to_frame("count"))

print("\nBCN image_type distribution:")
display(bcn["image_type"].value_counts(dropna=False).to_frame("count"))

print("\nHAM diagnosis_3 distribution:")
display(ham["diagnosis_3"].value_counts(dropna=False).to_frame("count").head(30))

print("\nBCN diagnosis_3 distribution:")
display(bcn["diagnosis_3"].value_counts(dropna=False).to_frame("count").head(30))



HAM unique/missing summary:


,column,n_unique,n_missing
0,age_approx,17,383
1,anatom_site_general,7,2162
2,diagnosis_confirm_type,4,0
3,harmonized_label,6,409
4,image_exists,1,0
5,image_type,1,1
6,isic_id,11720,0
7,lesion_id,8838,0
8,primary_status,3,0
9,raw_label,7,180



BCN unique/missing summary:


,column,n_unique,n_missing
0,age_approx,18,121
1,anatom_site_general,6,297
2,diagnosis_confirm_type,2,3506
3,harmonized_label,6,2813
4,image_exists,1,0
5,image_type,1,0
6,isic_id,18946,0
7,lesion_id,5440,0
8,primary_status,5,0
9,raw_label,10,1307



HAM image_type distribution:


...

,count
image_type,
dermoscopic,18946



HAM diagnosis_3 distribution:


,count
diagnosis_3,
Nevus,7737
Pigmented benign keratosis,1338
"Melanoma, NOS",1305
Basal cell carcinoma,622
"Squamous cell carcinoma, NOS",229
NaN,180
Dermatofibroma,160
Solar or actinic keratosis,149



BCN diagnosis_3 distribution:


,count
diagnosis_3,
Nevus,5647
"Melanoma, NOS",4003
Basal cell carcinoma,3676
NaN,1307
Seborrheic keratosis,1268
Solar or actinic keratosis,1088
Melanoma metastasis,633
"Squamous cell carcinoma, NOS",559
Scar,314


In [5]:

# =========================================
# 4) Check for missing / duplicate ISIC IDs
# =========================================

for name, df in [("HAM10000", ham), ("BCN20000", bcn)]:
    print(f"\n{name}")
    print("-" * len(name))
    print("Missing isic_id:", int(df["isic_id"].isna().sum()))
    print("Duplicate isic_id rows:", int(df["isic_id"].duplicated().sum()))
    if df["isic_id"].duplicated().any():
        display(df[df["isic_id"].duplicated(keep=False)].sort_values("isic_id").head(20))



HAM10000
--------
Missing isic_id: 0
Duplicate isic_id rows: 0

BCN20000
--------
Missing isic_id: 0
Duplicate isic_id rows: 0


In [6]:

# ==================================================
# 5) Check images that are missing on disk
# ==================================================

for name, df in [("HAM10000", ham), ("BCN20000", bcn)]:
    print(f"\n{name} missing image files:", int((~df["image_exists"]).sum()))
    if (~df["image_exists"]).any():
        display(df.loc[~df["image_exists"], ["isic_id", "source_dataset", "raw_label"]].head(20))



HAM10000 missing image files: 0

BCN20000 missing image files: 0


In [7]:

# ====================================================
# 6) Label cleaning report for the primary 7-class setup
# ====================================================

for name, df in [("HAM10000", ham), ("BCN20000", bcn)]:
    print(f"\n{'='*80}\n{name} label cleaning report\n{'='*80}")
    print("\nRaw diagnosis_3 counts:")
    display(df["raw_label"].value_counts(dropna=False).to_frame("count"))

    print("\nPrimary status counts:")
    display(df["primary_status"].value_counts(dropna=False).to_frame("count"))

    print("\nHarmonized 7-class counts (kept only):")
    display(df.loc[df["keep_for_primary_experiment"], "harmonized_label"].value_counts(dropna=False).to_frame("count"))



HAM10000 label cleaning report

Raw diagnosis_3 counts:


,count
raw_label,
Nevus,7737
Pigmented benign keratosis,1338
"Melanoma, NOS",1305
Basal cell carcinoma,622
"Squamous cell carcinoma, NOS",229
NaN,180
Dermatofibroma,160
Solar or actinic keratosis,149



Primary status counts:


,count
primary_status,
keep,11311
exclude_scc,229
exclude_missing_diagnosis,180



Harmonized 7-class counts (kept only):


...

,count
raw_label,
Nevus,5647
"Melanoma, NOS",4003
Basal cell carcinoma,3676
NaN,1307
Seborrheic keratosis,1268
Solar or actinic keratosis,1088
Melanoma metastasis,633
"Squamous cell carcinoma, NOS",559
Scar,314



Primary status counts:


,count
primary_status,
keep,16133
exclude_missing_diagnosis,1307
exclude_melanoma_metastasis,633
exclude_scc,559
exclude_scar,314



Harmonized 7-class counts (kept only):


,count
harmonized_label,
NV,5647
MEL,4003
BCC,3676
BKL,1551
AKIEC,1088
DF,168


In [8]:

# =====================================================
# 7) Lesion-level view: multiple images per lesion
# =====================================================

def lesion_summary(df, name):
    lesion_counts = df.groupby("lesion_id", dropna=False)["isic_id"].nunique().rename("n_images").reset_index()
    print(f"\n{name}: lesion-level summary")
    print("Rows:", len(df))
    print("Unique lesions:", df["lesion_id"].nunique(dropna=True))
    print("Rows with missing lesion_id:", int(df["lesion_id"].isna().sum()))
    print("\nImages per lesion distribution:")
    display(lesion_counts["n_images"].value_counts().sort_index().to_frame("n_lesions"))
    print("\nTop lesions with most images:")
    display(
        df.groupby("lesion_id", dropna=False)
          .agg(
              n_images=("isic_id", "nunique"),
              labels=("harmonized_label", lambda s: sorted(set([x for x in s.dropna().tolist()]))),
              sources=("source_dataset", lambda s: sorted(set(s.tolist())))
          )
          .sort_values("n_images", ascending=False)
          .head(20)
          .reset_index()
    )

lesion_summary(ham, "HAM10000")
lesion_summary(bcn, "BCN20000")



HAM10000: lesion-level summary
Rows: 11720
Unique lesions: 8838
Rows with missing lesion_id: 0

Images per lesion distribution:


,n_lesions
n_images,
1,6617
2,1619
3,556
4,37
5,5
6,4



Top lesions with most images:


,lesion_id,n_images,labels,sources
0,IL_2755785,6,[BKL],[HAM10000]
1,IL_1795129,6,[NV],[HAM10000]
2,IL_1017066,6,[BKL],[HAM10000]
3,IL_2334846,6,[MEL],[HAM10000]
4,IL_3857855,5,[NV],[HAM10000]
5,IL_4551923,5,[NV],[HAM10000]
6,IL_8711548,5,[BKL],[HAM10000]
7,IL_1446570,5,[MEL],[HAM10000]
8,IL_4222523,5,[BKL],[HAM10000]
9,IL_1075649,4,[MEL],[HAM10000]



BCN20000: lesion-level summary
Rows: 18946
Unique lesions: 5440
Rows with missing lesion_id: 0

Images per lesion distribution:


,n_lesions
n_images,
1,815
2,1583
3,1184
4,662
5,396
6,284
7,166
8,84
9,79



Top lesions with most images:


,lesion_id,n_images,labels,sources
0,IL_2969638,31,[MEL],[BCN20000]
1,IL_5161836,31,[MEL],[BCN20000]
2,IL_4702939,27,[MEL],[BCN20000]
3,IL_4163485,27,[MEL],[BCN20000]
4,IL_9292662,26,[MEL],[BCN20000]
5,IL_2894588,24,[],[BCN20000]
6,IL_6987249,24,[MEL],[BCN20000]
7,IL_3937356,24,[MEL],[BCN20000]
8,IL_0087971,23,[MEL],[BCN20000]
9,IL_2857034,22,[MEL],[BCN20000]


In [9]:

# ======================================================================
# 8) Build merged master metadata table (before duplicate image hashing)
# ======================================================================

master = pd.concat([ham, bcn], ignore_index=True, sort=False)

print("Master shape:", master.shape)
print("\nMaster source distribution:")
display(master["source_dataset"].value_counts(dropna=False).to_frame("count"))

print("\nMaster keep_for_primary_experiment:")
display(master["keep_for_primary_experiment"].value_counts(dropna=False).to_frame("count"))

print("\nMaster harmonized label counts (kept only):")
display(master.loc[master["keep_for_primary_experiment"], "harmonized_label"].value_counts(dropna=False).to_frame("count"))

print("\nCross-tab: source x harmonized label (kept only)")
display(pd.crosstab(
    master.loc[master["keep_for_primary_experiment"], "source_dataset"],
    master.loc[master["keep_for_primary_experiment"], "harmonized_label"],
    margins=True
))


Master shape: (30666, 29)

Master source distribution:


,count
source_dataset,
BCN20000,18946
HAM10000,11720



Master keep_for_primary_experiment:


,count
keep_for_primary_experiment,
True,27443
False,3223



Master harmonized label counts (kept only):


,count
harmonized_label,
NV,13383
MEL,5308
BCC,4298
BKL,2889
AKIEC,1237
DF,328



Cross-tab: source x harmonized label (kept only)


harmonized_label,AKIEC,BCC,BKL,DF,MEL,NV,All
source_dataset,,,,,,,
BCN20000,1088,3676,1551,168,4003,5647,16133
HAM10000,149,622,1338,160,1305,7736,11310
All,1237,4298,2889,328,5308,13383,27443


## Exact duplicate hashing

This step computes **SHA256** hashes of the image files. It helps detect:

- exact duplicate images within HAM
- exact duplicate images within BCN
- exact duplicate images across HAM and BCN

This does **not** catch near-duplicates. It only catches byte-identical files.


In [10]:

# ==========================================================
# 9) Compute exact SHA256 hashes for images that exist
# ==========================================================

HASH_CSV = OUT_DIR / "image_hashes_sha256.csv"

compute_hashes = True  # set False if you want to skip temporarily

if compute_hashes:
    hash_records = []
    rows_to_hash = master.loc[master["image_exists"], ["isic_id", "source_dataset", "image_path", "lesion_id", "harmonized_label", "keep_for_primary_experiment"]].copy()

    print("Number of images to hash:", len(rows_to_hash))

    for i, row in enumerate(rows_to_hash.itertuples(index=False), start=1):
        img_path = Path(row.image_path)
        try:
            sha = sha256_file(img_path)
        except Exception as e:
            sha = np.nan
            print(f"[WARN] Could not hash {img_path}: {e}")

        hash_records.append({
            "isic_id": row.isic_id,
            "source_dataset": row.source_dataset,
            "image_path": str(img_path),
            "lesion_id": row.lesion_id,
            "harmonized_label": row.harmonized_label,
            "keep_for_primary_experiment": row.keep_for_primary_experiment,
            "sha256": sha,
        })

        if i % 500 == 0 or i == len(rows_to_hash):
            print(f"Hashed {i}/{len(rows_to_hash)} images")

    hashes = pd.DataFrame(hash_records)
    hashes.to_csv(HASH_CSV, index=False)
    print("\nSaved hashes to:", HASH_CSV)
else:
    hashes = pd.read_csv(HASH_CSV)
    print("Loaded existing hashes from:", HASH_CSV)

print("\nHash table shape:", hashes.shape)
display(hashes.head())


Number of images to hash: 30666
Hashed 500/30666 images
Hashed 1000/30666 images
Hashed 1500/30666 images
Hashed 2000/30666 images
Hashed 2500/30666 images
Hashed 3000/30666 images
Hashed 3500/30666 images
Hashed 4000/30666 images
Hashed 4500/30666 images
Hashed 5000/30666 images
Hashed 5500/30666 images
Hashed 6000/30666 images
Hashed 6500/30666 images
Hashed 7000/30666 images
Hashed 7500/30666 images
Hashed 8000/30666 images
Hashed 8500/30666 images
Hashed 9000/30666 images
Hashed 9500/30666 images
Hashed 10000/30666 images
Hashed 10500/30666 images
Hashed 11000/30666 images
Hashed 11500/30666 images
Hashed 12000/30666 images
Hashed 12500/30666 images
Hashed 13000/30666 images
Hashed 13500/30666 images
Hashed 14000/30666 images
Hashed 14500/30666 images
Hashed 15000/30666 images
Hashed 15500/30666 images
Hashed 16000/30666 images
Hashed 16500/30666 images
Hashed 17000/30666 images
Hashed 17500/30666 images
Hashed 18000/30666 images
Hashed 18500/30666 images
Hashed 19000/30666 images


,isic_id,source_dataset,image_path,lesion_id,harmonized_label,keep_for_primary_experiment,sha256
0,ISIC_0024306,HAM10000,../data/HAM10k/ISIC_0024306.jpg,IL_7252831,NV,True,df3c09f55e58236d31c0d763d1646a73019f13d4c23496b13d764d19288e2cfb
1,ISIC_0024307,HAM10000,../data/HAM10k/ISIC_0024307.jpg,IL_6125741,NV,True,68715128720d233e42e70e8b81053e87ffcb1737b025ca03501edd6d11f42ab4
2,ISIC_0024308,HAM10000,../data/HAM10k/ISIC_0024308.jpg,IL_3692653,NV,True,44d463dc495e08894e74769b088f2c827f9dac10b4d05ececcb32ee3aa285f79
3,ISIC_0024309,HAM10000,../data/HAM10k/ISIC_0024309.jpg,IL_0959663,NV,True,8822663834a998e1a8020ced54dd05f4eeca49d888214fde6c4e55055752a35a
4,ISIC_0024310,HAM10000,../data/HAM10k/ISIC_0024310.jpg,IL_8194852,MEL,True,f79b817ee580c6958219143f63e31f0f0456b42650cea33a48638d868574ec0a


In [11]:

# =======================================================
# 10) Exact duplicate report based on SHA256
# =======================================================

if "hashes" not in globals():
    hashes = pd.read_csv(HASH_CSV)

dup_exact = hashes.loc[hashes["sha256"].notna()].groupby("sha256").filter(lambda g: len(g) > 1).copy()

print("Number of rows involved in exact duplicate groups:", len(dup_exact))
print("Number of exact duplicate groups:", dup_exact["sha256"].nunique() if len(dup_exact) else 0)

if len(dup_exact):
    dup_summary = (
        dup_exact.groupby("sha256")
        .agg(
            n_images=("isic_id", "count"),
            datasets=("source_dataset", lambda s: sorted(set(s.tolist()))),
            isic_ids=("isic_id", lambda s: sorted(s.tolist())),
            lesion_ids=("lesion_id", lambda s: sorted(set([x for x in s.dropna().tolist()]))),
            labels=("harmonized_label", lambda s: sorted(set([x for x in s.dropna().tolist()])))
        )
        .sort_values("n_images", ascending=False)
        .reset_index()
    )
    display(dup_summary.head(20))

    DUP_EXACT_CSV = OUT_DIR / "exact_duplicate_groups.csv"
    dup_summary.to_csv(DUP_EXACT_CSV, index=False)
    print("\nSaved exact duplicate group report to:", DUP_EXACT_CSV)
else:
    print("No exact duplicates found.")


Number of rows involved in exact duplicate groups: 0
Number of exact duplicate groups: 0
No exact duplicates found.


## Conservative keep rule for the first clean pass

For the first baseline, we will **not automatically delete duplicate images**.  
Instead, we will create flags so you can inspect them first.

Recommended later rule:
- if exact duplicates exist, keep only one representative **per duplicate cluster**
- if duplicates come from the same lesion, always keep them in the **same split group**
- if duplicates appear across datasets, inspect them manually before final split generation


In [12]:

# =======================================================
# 11) Merge duplicate flags back into master metadata
# =======================================================

if "hashes" not in globals():
    hashes = pd.read_csv(HASH_CSV)

master = master.merge(
    hashes[["isic_id", "sha256"]],
    on="isic_id",
    how="left"
)

sha_counts = master["sha256"].value_counts(dropna=True)
master["exact_duplicate_group_size"] = master["sha256"].map(sha_counts).fillna(1).astype(int)
master["has_exact_duplicate"] = master["exact_duplicate_group_size"] > 1

print("Master with duplicate flags:")
display(master[[
    "isic_id", "source_dataset", "lesion_id", "harmonized_label",
    "keep_for_primary_experiment", "sha256", "exact_duplicate_group_size", "has_exact_duplicate"
]].head(10))

print("\nRows with exact duplicates:", int(master["has_exact_duplicate"].sum()))


Master with duplicate flags:


,isic_id,source_dataset,lesion_id,harmonized_label,keep_for_primary_experiment,sha256,exact_duplicate_group_size,has_exact_duplicate
0,ISIC_0024306,HAM10000,IL_7252831,NV,True,df3c09f55e58236d31c0d763d1646a73019f13d4c23496b13d764d19288e2cfb,1,False
1,ISIC_0024307,HAM10000,IL_6125741,NV,True,68715128720d233e42e70e8b81053e87ffcb1737b025ca03501edd6d11f42ab4,1,False
2,ISIC_0024308,HAM10000,IL_3692653,NV,True,44d463dc495e08894e74769b088f2c827f9dac10b4d05ececcb32ee3aa285f79,1,False
3,ISIC_0024309,HAM10000,IL_0959663,NV,True,8822663834a998e1a8020ced54dd05f4eeca49d888214fde6c4e55055752a35a,1,False
4,ISIC_0024310,HAM10000,IL_8194852,MEL,True,f79b817ee580c6958219143f63e31f0f0456b42650cea33a48638d868574ec0a,1,False
5,ISIC_0024311,HAM10000,IL_9035157,NV,True,73addcf8e23e4b208bee2fd50f446daff1f0e8fda2f1832c9a5d5100c854a407,1,False
6,ISIC_0024312,HAM10000,IL_3471258,BKL,True,ac3cddf42e2f3b733e2cd5eee99695ca812d989b727638fcb6ec40fdb0e86a2f,1,False
7,ISIC_0024313,HAM10000,IL_9808449,MEL,True,3eeeb5371493cd976b32cf3ded3c9df10d97da26f33c3f1a32f07e92f54ff539,1,False
8,ISIC_0024314,HAM10000,IL_4137430,NV,True,ee346ca59a4ffa10fe1efd3d03ee79eb1cfadcaf4c8b281ac3e0737d3fd0a02a,1,False
9,ISIC_0024315,HAM10000,IL_9534602,MEL,True,9d3a4f87760391b31c21f222403745418fd4a15b4f01d7c765d7763b6a882cc0,1,False



Rows with exact duplicates: 0


In [13]:

# ==========================================================
# 12) Final cleaned views for the primary 7-class experiment
# ==========================================================

ham_clean = master.query("source_dataset == 'HAM10000'").copy()
bcn_clean = master.query("source_dataset == 'BCN20000'").copy()

primary_clean = master.loc[master["keep_for_primary_experiment"]].copy()

print("HAM clean shape:", ham_clean.shape)
print("BCN clean shape:", bcn_clean.shape)
print("Primary clean shape (kept rows only):", primary_clean.shape)

print("\nPrimary clean label counts:")
display(primary_clean["harmonized_label"].value_counts().to_frame("count"))

print("\nPrimary clean source x label:")
display(pd.crosstab(primary_clean["source_dataset"], primary_clean["harmonized_label"], margins=True))


HAM clean shape: (11720, 32)
BCN clean shape: (18946, 32)
Primary clean shape (kept rows only): (27443, 32)

Primary clean label counts:


,count
harmonized_label,
NV,13383
MEL,5308
BCC,4298
BKL,2889
AKIEC,1237
DF,328



Primary clean source x label:


harmonized_label,AKIEC,BCC,BKL,DF,MEL,NV,All
source_dataset,,,,,,,
BCN20000,1088,3676,1551,168,4003,5647,16133
HAM10000,149,622,1338,160,1305,7736,11310
All,1237,4298,2889,328,5308,13383,27443


In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

def add_sha256_column(df, path_col="image_path", out_col="sha256"):
    df = df.copy()
    hashes = []

    print(f"Computing SHA256 for {len(df)} rows...")
    for i, p in enumerate(df[path_col]):
        if pd.isna(p):
            hashes.append(np.nan)
            continue

        p = Path(p)
        if not p.exists():
            hashes.append(np.nan)
            continue

        hashes.append(sha256_file(p))

        if (i + 1) % 1000 == 0 or (i + 1) == len(df):
            print(f"  processed {i+1}/{len(df)}")

    df[out_col] = hashes
    return df

In [16]:
ham_df = add_sha256_column(ham_clean)
bcn_df = add_sha256_column(bcn_clean)

master_df = pd.concat([ham_df, bcn_df], ignore_index=True)
print("master shape with hashes:", master_df.shape)

Computing SHA256 for 11720 rows...
  processed 1000/11720
  processed 2000/11720
  processed 3000/11720
  processed 4000/11720
  processed 5000/11720
  processed 6000/11720
  processed 7000/11720
  processed 8000/11720
  processed 9000/11720
  processed 10000/11720
  processed 11000/11720
  processed 11720/11720
Computing SHA256 for 18946 rows...
  processed 1000/18946
  processed 2000/18946
  processed 3000/18946
  processed 4000/18946
  processed 5000/18946
  processed 6000/18946
  processed 7000/18946
  processed 8000/18946
  processed 9000/18946
  processed 10000/18946
  processed 11000/18946
  processed 12000/18946
  processed 13000/18946
  processed 14000/18946
  processed 15000/18946
  processed 16000/18946
  processed 17000/18946
  processed 18000/18946
  processed 18946/18946
master shape with hashes: (30666, 32)


In [17]:
def summarize_hash_duplicates(df, hash_col="sha256", name="dataset"):
    print(f"\n{'='*80}")
    print(f"{name}: SHA256 duplicate summary")
    print(f"{'='*80}")

    n_missing = df[hash_col].isna().sum()
    n_unique = df[hash_col].nunique(dropna=True)

    dup_mask = df[hash_col].duplicated(keep=False) & df[hash_col].notna()
    dup_df = df.loc[dup_mask].copy()

    print("rows:", len(df))
    print("missing hashes:", int(n_missing))
    print("unique hashes:", int(n_unique))
    print("rows involved in duplicate hashes:", int(dup_mask.sum()))
    print("number of duplicate hash groups:", int(dup_df[hash_col].nunique()))

    if len(dup_df) > 0:
        print("\nTop duplicate groups:")
        preview = (
            dup_df.groupby(hash_col)
            .agg(
                n_rows=("isic_id", "size"),
                datasets=("source_dataset", lambda x: sorted(set(x))),
                isic_ids=("isic_id", lambda x: list(x)[:10]),
                lesion_ids=("lesion_id", lambda x: list(pd.Series(x).dropna().unique())[:10]),
            )
            .sort_values("n_rows", ascending=False)
            .head(20)
            .reset_index()
        )
        display(preview)
    else:
        print("No exact duplicate hashes found.")

summarize_hash_duplicates(ham_df, name="HAM10000")
summarize_hash_duplicates(bcn_df, name="BCN20000")
summarize_hash_duplicates(master_df, name="MASTER (HAM + BCN)")


HAM10000: SHA256 duplicate summary
rows: 11720
missing hashes: 0
unique hashes: 11720
rows involved in duplicate hashes: 0
number of duplicate hash groups: 0
No exact duplicate hashes found.

BCN20000: SHA256 duplicate summary
rows: 18946
missing hashes: 0
unique hashes: 18946
rows involved in duplicate hashes: 0
number of duplicate hash groups: 0
No exact duplicate hashes found.

MASTER (HAM + BCN): SHA256 duplicate summary
rows: 30666
missing hashes: 0
unique hashes: 30666
rows involved in duplicate hashes: 0
number of duplicate hash groups: 0
No exact duplicate hashes found.


In [18]:
def cross_dataset_hash_duplicates(df, hash_col="sha256"):
    dup_mask = df[hash_col].duplicated(keep=False) & df[hash_col].notna()
    dup_df = df.loc[dup_mask].copy()

    if dup_df.empty:
        print("No duplicate hashes at all.")
        return dup_df

    cross = (
        dup_df.groupby(hash_col)
        .agg(
            n_rows=("isic_id", "size"),
            n_datasets=("source_dataset", lambda x: len(set(x))),
            datasets=("source_dataset", lambda x: sorted(set(x))),
            isic_ids=("isic_id", lambda x: list(x)),
            lesion_ids=("lesion_id", lambda x: list(pd.Series(x).dropna().unique())),
        )
        .reset_index()
    )

    cross = cross[cross["n_datasets"] > 1].sort_values("n_rows", ascending=False)

    print(f"Cross-dataset duplicate hash groups: {len(cross)}")
    if len(cross) > 0:
        display(cross.head(30))
    else:
        print("No exact duplicates found across datasets.")

    return cross

cross_hash_df = cross_dataset_hash_duplicates(master_df)

No duplicate hashes at all.


In [20]:
def lesion_label_conflict_report(df, dataset_name="dataset"):
    kept = df[df["keep_for_primary_experiment"]].copy()

    lesion_summary = (
        kept.groupby("lesion_id")
        .agg(
            n_images=("isic_id", "size"),
            n_labels=("harmonized_label", lambda x: x.nunique(dropna=True)),
            labels=("harmonized_label", lambda x: sorted(pd.Series(x).dropna().unique())),
        )
        .reset_index()
    )

    conflicts = lesion_summary[lesion_summary["n_labels"] > 1].copy()

    print(f"\n{'='*80}")
    print(f"{dataset_name}: lesion label conflict report")
    print(f"{'='*80}")
    print("kept rows:", len(kept))
    print("unique lesions:", kept["lesion_id"].nunique())
    print("lesions with >1 kept label:", len(conflicts))

    if len(conflicts) > 0:
        display(conflicts.sort_values(["n_labels", "n_images"], ascending=False).head(30))
    else:
        print("No lesion-level label conflicts found.")

    return conflicts

ham_conflicts = lesion_label_conflict_report(ham_df, "HAM10000")
bcn_conflicts = lesion_label_conflict_report(bcn_df, "BCN20000")
master_conflicts = lesion_label_conflict_report(master_df, "MASTER")


HAM10000: lesion label conflict report
kept rows: 11310
unique lesions: 8542
lesions with >1 kept label: 0
No lesion-level label conflicts found.

BCN20000: lesion label conflict report
kept rows: 16133
unique lesions: 4543
lesions with >1 kept label: 0
No lesion-level label conflicts found.

MASTER: lesion label conflict report
kept rows: 27443
unique lesions: 13085
lesions with >1 kept label: 0
No lesion-level label conflicts found.


In [ ]:

# =======================================================
# 13) Save outputs
# =======================================================

ham_out = OUT_DIR / "ham10000_clean_metadata.csv"
bcn_out = OUT_DIR / "bcn20000_clean_metadata.csv"
master_out = OUT_DIR / "master_clean_metadata.csv"
primary_out = OUT_DIR / "primary6_keep_only_metadata.csv"

ham_clean.to_csv(ham_out, index=False)
bcn_clean.to_csv(bcn_out, index=False)
master.to_csv(master_out, index=False)
primary_clean.to_csv(primary_out, index=False)

print("Saved:")
print(" -", ham_out)
print(" -", bcn_out)
print(" -", master_out)
print(" -", primary_out)


Saved:
 - ../data/processed/ham10000_clean_metadata.csv
 - ../data/processed/bcn20000_clean_metadata.csv
 - ../data/processed/master_clean_metadata.csv
 - ../data/processed/primary7_keep_only_metadata.csv


## Suggested next notebook

After this notebook, the next notebook should do:

1. **Grouped split generation**
   - group by `patient_id` if available
   - otherwise `lesion_id`
   - otherwise duplicate cluster / `isic_id`

2. **Split design**
   - `train_base.csv`: HAM + MSK
   - `val_base.csv`
   - `train_bcn_ft.csv`
   - `val_bcn_ft.csv`
   - `test_bcn.csv`

3. **Optional near-duplicate pass**
   - perceptual hash
   - embedding similarity shortlist
   - manual review of suspicious cross-dataset pairs

For your thesis, this notebook is the **clean audit + metadata foundation**.
